# Episode Dataset Audit - 7 IDSSE Matches

Notebook ini mengaudit seluruh active possession change sebelum dataset GNN dimaterialisasi.

Konfigurasi audit:
- label positif: losing team regain dalam 5 detik;
- window struktur: T0-2 detik sampai T0+2 detik;
- sampling tracking: 25 Hz, sehingga expected window adalah 101 frame;
- node strict: 11 pemain losing team, 11 opponent, dan bola;
- edge: directed kNN dengan k=4.

Window ini hanya digunakan untuk audit konsistensi. Window training dan penanganan label leakage belum ditetapkan. Sampel 10-vs-11 dicatat sebagai tidak strict-valid, bukan langsung dianggap data rusak.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.idsse_episode_dataset import (
    EpisodeSampleConfig,
    audit_all_matches,
    build_episode_sample,
    load_match,
)

DATA_DIR = PROJECT_ROOT / "data" / "sportec_idsse"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MATCH_IDS = [
    "J03WPY",
    "J03WMX",
    "J03WN1",
    "J03WOH",
    "J03WOY",
    "J03WQQ",
    "J03WR9",
]
CONFIG = EpisodeSampleConfig(
    pre_seconds=2.0,
    post_seconds=2.0,
    regain_horizon_seconds=5.0,
    frame_rate_hz=25,
    k_neighbors=4,
)

print("Data directory:", DATA_DIR)
print("Expected frames per sample:", CONFIG.expected_frame_count)

Data directory: c:\VAEP-Track\data\sportec_idsse
Expected frames per sample: 101


In [2]:
candidates_df, audit_df = audit_all_matches(
    data_dir=DATA_DIR,
    match_ids=MATCH_IDS,
    config=CONFIG,
    verbose=True,
)

episode_manifest = candidates_df.merge(
    audit_df.drop(columns=["label_regain_5s"]),
    on=[
        "candidate_id",
        "match_id",
        "period_id",
        "loss_frame",
        "loss_timestamp",
    ],
    how="left",
    validate="one_to_one",
)

candidate_path = OUTPUT_DIR / "turnover_candidates.csv"
audit_path = OUTPUT_DIR / "episode_sample_audit.csv"
manifest_path = OUTPUT_DIR / "episode_manifest.csv"

candidates_df.to_csv(candidate_path, index=False)
audit_df.to_csv(audit_path, index=False)
episode_manifest.to_csv(manifest_path, index=False)

print("Candidate rows:", len(candidates_df))
print("Audit rows:", len(audit_df))
print("Manifest rows:", len(episode_manifest))
print("Saved manifest:", manifest_path)

J03WPY: candidates=324, positive=148, strict_valid=323
J03WMX: candidates=408, positive=211, strict_valid=408
J03WN1: candidates=394, positive=204, strict_valid=33
J03WOH: candidates=301, positive=138, strict_valid=298
J03WOY: candidates=344, positive=171, strict_valid=342
J03WQQ: candidates=304, positive=121, strict_valid=195
J03WR9: candidates=278, positive=112, strict_valid=274
Candidate rows: 2353
Audit rows: 2353
Manifest rows: 2353
Saved manifest: c:\VAEP-Track\data\processed\episode_manifest.csv


In [3]:
summary_by_match = (
    episode_manifest
    .groupby("match_id")
    .agg(
        candidates=("candidate_id", "count"),
        positive_regain_5s=("label_regain_5s", "sum"),
        strict_valid=("strict_valid", "sum"),
    )
    .reset_index()
)
summary_by_match["negative_no_regain_5s"] = (
    summary_by_match["candidates"]
    - summary_by_match["positive_regain_5s"]
)
summary_by_match["strict_invalid"] = (
    summary_by_match["candidates"]
    - summary_by_match["strict_valid"]
)
summary_by_match["strict_valid_rate"] = (
    summary_by_match["strict_valid"]
    / summary_by_match["candidates"]
).round(4)

valid_by_label = (
    episode_manifest
    .groupby(["label_regain_5s", "strict_valid"])
    .size()
    .rename("samples")
    .reset_index()
)

failure_reason_counts = (
    audit_df.loc[~audit_df["strict_valid"], "failure_reasons"]
    .str.split("|")
    .explode()
    .loc[lambda values: values.ne("")]
    .value_counts()
    .rename_axis("failure_reason")
    .reset_index(name="samples")
)

roster_distribution = (
    audit_df
    .groupby(["match_id", "active_player_count"])
    .size()
    .rename("samples")
    .reset_index()
)

print("Total candidates:", len(episode_manifest))
print("Positive regain <= 5s:", int(episode_manifest["label_regain_5s"].sum()))
print("Negative:", int((episode_manifest["label_regain_5s"] == 0).sum()))
print("Strict valid:", int(episode_manifest["strict_valid"].sum()))
print("Strict invalid:", int((~episode_manifest["strict_valid"]).sum()))

display(summary_by_match)
display(valid_by_label)
display(failure_reason_counts)
display(roster_distribution)

Total candidates: 2353
Positive regain <= 5s: 1105
Negative: 1248
Strict valid: 1873
Strict invalid: 480


,match_id,candidates,positive_regain_5s,strict_valid,negative_no_regain_5s,strict_invalid,strict_valid_rate
0,J03WMX,408,211,408,197,0,1.0000
1,J03WN1,394,204,33,190,361,0.0838
2,J03WOH,301,138,298,163,3,0.9900
3,J03WOY,344,171,342,173,2,0.9942
4,J03WPY,324,148,323,176,1,0.9969
5,J03WQQ,304,121,195,183,109,0.6414
6,J03WR9,278,112,274,166,4,0.9856


,label_regain_5s,strict_valid,samples
0,0,False,240
1,0,True,1008
2,1,False,240
3,1,True,865


,failure_reason,samples
0,active_player_count_not_22,466
1,team_split_not_11_11,466
2,incomplete_window,14


,match_id,active_player_count,samples
0,J03WMX,22,408
1,J03WN1,19,11
2,J03WN1,20,16
3,J03WN1,21,334
4,J03WN1,22,33
5,J03WOH,22,301
6,J03WOY,22,344
7,J03WPY,22,324
8,J03WQQ,20,3
9,J03WQQ,21,102


In [4]:
selected_event = (
    episode_manifest[
        (episode_manifest["match_id"] == "J03WMX")
        & episode_manifest["strict_valid"]
        & (episode_manifest["label_regain_5s"] == 1)
    ]
    .iloc[0]
)

sample_tracking, sample_events = load_match(
    DATA_DIR,
    selected_event["match_id"],
)
sample = build_episode_sample(
    event=selected_event,
    tracking=sample_tracking,
    events=sample_events,
    config=CONFIG,
    materialize_graphs=True,
)

edge_counts = [edges.shape[1] for edges in sample["edge_index"]]

print("Candidate ID:", sample["candidate_id"])
print("Label:", sample["label"])
print("Strict valid:", sample["audit"]["strict_valid"])
print("Node feature tensor:", sample["node_features"].shape)
print("Node mask:", sample["node_mask"].shape)
print("Frame count:", len(sample["frame_ids"]))
print("Edge count range:", min(edge_counts), "to", max(edge_counts))
print("Relative time:", sample["relative_times"][0], "to", sample["relative_times"][-1])

Candidate ID: J03WMX_P1_F12051
Label: 1
Strict valid: True
Node feature tensor: (101, 23, 6)
Node mask: (101, 23)
Frame count: 101
Edge count range: 92 to 92
Relative time: -2.0 to 2.0


In [5]:
import torch
from torch_geometric.data import Data

pyg_sequence = []
for frame_idx in range(len(sample["frame_ids"])):
    pyg_sequence.append(Data(
        x=torch.tensor(
            sample["node_features"][frame_idx],
            dtype=torch.float32,
        ),
        edge_index=torch.tensor(
            sample["edge_index"][frame_idx],
            dtype=torch.long,
        ),
        edge_attr=torch.tensor(
            sample["edge_attr"][frame_idx],
            dtype=torch.float32,
        ),
        node_mask=torch.tensor(
            sample["node_mask"][frame_idx],
            dtype=torch.bool,
        ),
        y=torch.tensor([sample["label"]], dtype=torch.long),
    ))

print("PyG sequence length:", len(pyg_sequence))
print("First graph:", pyg_sequence[0])
print("Last graph:", pyg_sequence[-1])

PyG sequence length: 101
First graph: Data(x=[23, 6], edge_index=[2, 92], edge_attr=[92, 1], y=[1], node_mask=[23])
Last graph: Data(x=[23, 6], edge_index=[2, 92], edge_attr=[92, 1], y=[1], node_mask=[23])
